# 1.3) NumPy

Numpy is the foundation of scientific Python: a dense, typed n-dimensional array (`ndarray`) and a large set of operations that act on whole arrays at once, without python-level loops. This notebook works with one example of a small two-dimensional temperature field on a latitude–longitude grid to cover array creation, dtype and shape, indexing and masking, vectorised math and broadcasting, reductions, and the handling of missing data. It closes with a generated-code bug that silently truncates results because of a dtype mistake.

:::{admonition} **Learning objectives**
:class: tip

- Create arrays from data and with constructors (`zeros`, `ones`, `arange`, `linspace`), and inspect their `shape` and `ndim`.
- Index, slice, and select elements with boolean masks.
- Replace element-wise loops with vectorised math and broadcasting.
- Call functions as a method (`arr.mean()`) or from numpy (`np.mean(arr)`).
- Reduce along chosen axes (`mean`, `min`, `max`, `argmax`, …), and reshape and stack arrays.
- Choose values with `np.where`, handle missing data with NaN-aware operations, and fill gaps with `np.interp`.
:::

## Creating arrays

An array is created from data — a nested list — or from a constructor. Every array carries a `shape` (its size along each axis) and an `ndim` (the number of axes).

In [1]:
import numpy as np

# a 2D field: 4 latitudes (rows) x 6 longitudes (cols), daily mean temp (°C)
temp_celsius = np.array([
    [ 5.2,  4.8,  6.1,  3.9,  2.7,  4.4],
    [ 1.3,  0.5, -0.8, -1.2,  0.9,  2.1],
    [-2.6, -3.1, -1.9,  0.2, -0.5,  1.1],
    [ 6.4,  7.0,  5.5,  8.1,  4.2,  5.8],
])

print(temp_celsius)
print("shape:", temp_celsius.shape, "| ndim:", temp_celsius.ndim)

[[ 5.2  4.8  6.1  3.9  2.7  4.4]
 [ 1.3  0.5 -0.8 -1.2  0.9  2.1]
 [-2.6 -3.1 -1.9  0.2 -0.5  1.1]
 [ 6.4  7.   5.5  8.1  4.2  5.8]]
shape: (4, 6) | ndim: 2


## Other ways to create arrays

Beyond a literal, numpy provides constructors for the patterns you need most often.

In [2]:
print(np.zeros((2, 3)))          # all zeros, given shape
print(np.ones(4))                # all ones
print(np.full((2, 2), 7.0))      # filled with a constant
print(np.arange(0, 10, 2))       # evenly spaced by step: [0 2 4 6 8]
print(np.linspace(0.0, 1.0, 5))  # n evenly spaced points: [0. 0.25 0.5 0.75 1. ]
print(np.random.random(3))       # 3 random floats in [0, 1)

[[0. 0. 0.]
 [0. 0. 0.]]
[1. 1. 1. 1.]
[[7. 7.]
 [7. 7.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[0.90479271 0.16591717 0.47786094]


:::{admonition} Going deeper: dtype and precision
:class: seealso dropdown
Every array also has a `dtype` — the element type — which fixes both its behaviour and its memory use. `float64` (double precision) is the default; `float32` halves the memory at the cost of precision.

```python
temp32 = temp_celsius.astype(np.float32)
print(temp_celsius.dtype, temp_celsius.nbytes, "bytes")   # float64, 192 bytes
print(temp32.dtype, temp32.nbytes, "bytes")               # float32, 96 bytes

# float32 is coarser: the rounding error in 0.1 + 0.2 disappears
print(np.float64(0.1) + np.float64(0.2))   # 0.30000000000000004
print(np.float32(0.1) + np.float32(0.2))   # 0.3
```

Watch the dtype when preallocating output arrays — it is the source of the silent bug later in this subchapter.
:::

## Indexing, slicing, and boolean masking

Indexing uses `[row, col]`; slicing selects sub-blocks; negative indices count from the end. A boolean *mask* is a same-shaped array of True/False that selects the matching elements.

```{mermaid}
flowchart LR
    A["2D array<br/>shape (4, 6)"]
    A -->|"arr[1, 2]"| E["single element"]
    A -->|"arr[1, :]"| R["one row (6,)"]
    A -->|"arr[:, 2]"| C["one column (4,)"]
    A -->|"arr[1:3, 2:4]"| S["sub-block (2, 2)"]
    A -->|"boolean mask"| M["arr[arr &lt; 0]:<br/>1D of matches"]
```

In [3]:
print(temp_celsius[0, 0])      # one element
print(temp_celsius[0])         # first row, all longitudes
print(temp_celsius[:, -1])     # last column, all latitudes
print(temp_celsius[1:3, 2:4])  # a 2x2 sub-block

# a boolean mask and what it selects
freezing = temp_celsius < 0.0
print("n freezing cells:", int(freezing.sum()))    # True counts as 1
print("freezing values:", temp_celsius[freezing])  # 1D of matches

5.2
[5.2 4.8 6.1 3.9 2.7 4.4]
[4.4 2.1 1.1 5.8]
[[-0.8 -1.2]
 [-1.9  0.2]]
n freezing cells: 6
freezing values: [-0.8 -1.2 -2.6 -3.1 -1.9 -0.5]


:::{admonition} Computational-thinking fundamental: think in arrays, not loops
:class: important
The central idea of numpy is *vectorisation*: express a computation as an operation on whole arrays rather than a loop over elements. `field + 273.15` converts every value at once. Vectorised code is shorter, far faster (the loop runs in compiled C), and closer to the mathematical statement of the problem. When you find yourself writing a python `for` loop over array elements, look for the array operation that replaces it.
:::

## Vectorised math and broadcasting

A single expression applies element-wise across an array. *Broadcasting* lets arrays of different but compatible shapes combine: a length-4 column vector can be stretched across all 6 longitudes.

```{mermaid}
flowchart LR
    A["array (4, 6)"] --> R["result (4, 6)"]
    B["column vector (4, 1)"] -->|"stretched across<br/>the 6 columns"| R
```

In [4]:
# vectorised: no python loop
temp_kelvin = temp_celsius + 273.15
print(temp_kelvin[0])           # first row, in kelvin

# broadcasting: a (4,1) column stretches across the 6 longitudes
lat_gradient_celsius = np.array([0.0, -1.5, -3.0, -4.5])   # colder toward the north
adjusted = temp_celsius + lat_gradient_celsius[:, None]    # (4,1) + (4,6) -> (4,6)
print(adjusted.shape)
print(adjusted)

[278.35 277.95 279.25 277.05 275.85 277.55]
(4, 6)
[[ 5.2  4.8  6.1  3.9  2.7  4.4]
 [-0.2 -1.  -2.3 -2.7 -0.6  0.6]
 [-5.6 -6.1 -4.9 -2.8 -3.5 -1.9]
 [ 1.9  2.5  1.   3.6 -0.3  1.3]]


## Calling functions on arrays

Most numpy operations are available two ways: as a *method* on the array (`arr.mean()`) or as a *function* in the numpy namespace (`np.mean(arr)`). They do the same thing — pick whichever reads more clearly.

In [5]:
print(temp_celsius.mean(), np.mean(temp_celsius))   # method and function agree
print(temp_celsius.sum(), np.sum(temp_celsius))

2.504166666666667 2.504166666666667
60.1 60.1


:::{admonition} Going deeper: other useful numpy functions
:class: seealso dropdown
numpy provides element-wise maths and helpers for tidying values. A few you will reach for often:

```python
a = np.array([1.234, -2.5, 9.876])
print(a.round(2))          # round to 2 decimals: [ 1.23 -2.5   9.88]
print(np.abs(a))           # absolute value
print(np.sqrt([1, 4, 9]))  # element-wise square root: [1. 2. 3.]
print(np.clip(a, 0, 5))    # limit values to the range [0, 5]
```

`round` is useful for readable output, but it is only for display — keep full precision inside a calculation.
:::

## Reductions, reshape, and stacking

A reduction collapses an axis: `axis=0` aggregates over latitudes (one result per longitude), `axis=1` over longitudes. `reshape` reorganises the same data; stacking combines arrays.

```{mermaid}
flowchart TD
    A["2D array (4, 6)"]
    A -->|"mean(axis=0)"| B["per-column (6,)"]
    A -->|"mean(axis=1)"| C["per-row (4,)"]
    A -->|"reshape(-1)"| D["flat (24,)"]
    A -->|"vstack with a (6,) row"| E["stacked (2, 6)"]
```

Common reductions all take an optional `axis`:

| Function | Returns |
| --- | --- |
| `sum`, `mean`, `std` | total, average, spread |
| `min`, `max` | smallest / largest value |
| `argmin`, `argmax` | index of the smallest / largest value |
| `cumsum`, `cumprod` | running total / product |

In [6]:
# a reduction collapses an axis: axis=0 over rows, axis=1 over columns
print("overall mean:", temp_celsius.mean())
print("mean per column (over rows):", temp_celsius.mean(axis=0))
print("mean per row (over columns):", temp_celsius.mean(axis=1))
print("min, max:", temp_celsius.min(), temp_celsius.max())
print("argmin, argmax (flat index):", temp_celsius.argmin(), temp_celsius.argmax())

# reshape: same 24 values, new shape; -1 infers the missing length
flat = temp_celsius.reshape(-1)
print("reshaped:", flat.shape)

# stacking: combine arrays along a new row axis
col_means = temp_celsius.mean(axis=0)
index_row = np.arange(6, dtype=float)
print("vstacked:", np.vstack([index_row, col_means]).shape)   # (2, 6)

overall mean: 2.504166666666667
mean per column (over rows): [2.575 2.3   2.225 2.75  1.825 3.35 ]
mean per row (over columns): [ 4.51666667  0.46666667 -1.13333333  6.16666667]
min, max: -3.1 8.1
argmin, argmax (flat index): 13 21
reshaped: (24,)
vstacked: (2, 6)


:::{admonition} Quick exercise: warmest latitude
:class: note
Compute the mean temperature of each latitude (each row), then use `argmax` to find the index of the warmest latitude.
:::

:::{admonition} Solution
:class: note dropdown
```python
row_means = temp_celsius.mean(axis=1)
print(row_means)
print(int(row_means.argmax()))
```
:::

## Choosing, missing data, and interpolation

`np.where` picks element-wise between two options. Missing data is represented by `NaN`; NaN-aware reductions (`np.nanmean`, …) skip it, while ordinary reductions propagate it. `np.interp` fills a 1D gap by linear interpolation.

In [7]:
# np.where(condition, a, b): element-wise choice
category = np.where(temp_celsius < 0.0, "freezing", "above")
print(category)

# missing data as NaN; nan-aware vs ordinary reduction
temp_with_gaps = temp_celsius.copy()
temp_with_gaps[0, 0] = np.nan
print("nanmean (skips gaps):", np.nanmean(temp_with_gaps))
print("plain mean is contaminated:", np.mean(temp_with_gaps))  # nan

# np.interp: fill a 1D gap by linear interpolation against an index
profile = temp_celsius[:, 0].copy()      # the first column, 4 latitudes
x = np.arange(profile.size)
known = np.array([0, 1, 3])              # pretend index 2 is missing
filled = np.interp(x, known, profile[known])
print(profile)
print(filled)                            # index 2 interpolated from its neighbours

[['above' 'above' 'above' 'above' 'above' 'above']
 ['above' 'above' 'freezing' 'freezing' 'above' 'above']
 ['freezing' 'freezing' 'freezing' 'above' 'freezing' 'above']
 ['above' 'above' 'above' 'above' 'above' 'above']]
nanmean (skips gaps): 2.38695652173913
plain mean is contaminated: nan
[ 5.2  1.3 -2.6  6.4]
[5.2  1.3  3.85 6.4 ]


## When generated code lies: a silent dtype truncation

ai assistants often preallocate an output array with `np.zeros_like`, which copies the *input's* dtype. If the input is integer, float results are silently truncated on assignment. Here a function computes anomalies (value minus the field mean) for an integer precipitation field.

In [8]:
def to_anomaly(field):
    # subtract the mean into a preallocated array (as an assistant returned it)
    result = np.zeros_like(field)        # inherits field's dtype!
    result[:] = field - field.mean()
    return result

precip_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])   # integer mm
print("input dtype:", precip_mm.dtype)
print(to_anomaly(precip_mm))

input dtype: int64
[[-2  0  2]
 [-1 -2  5]
 [ 0  1  0]]


:::{admonition} Diagnosis: the output inherited an integer dtype
:class: warning
The anomalies should be fractional, but every value is a whole number. `np.zeros_like(field)` produced an *integer* array because `field` is integer, and assigning float anomalies into it truncates each toward zero. The error is silent — no exception, just wrong numbers. The fix is to let numpy choose the result type, or to request `dtype=float` explicitly.
:::

In [9]:
def to_anomaly(field):
    # let numpy promote to float; no wrong-dtype preallocation
    return field - field.mean()

print(to_anomaly(precip_mm))
print("output dtype:", to_anomaly(precip_mm).dtype)

[[-2.77777778 -0.77777778  2.22222222]
 [-1.77777778 -2.77777778  5.22222222]
 [ 0.22222222  1.22222222 -0.77777778]]
output dtype: float64


:::{admonition} Going deeper: memory layout, views, and strides
:class: seealso dropdown
An array is a flat block of memory plus a `shape` and `strides` (the byte step along each axis). Slicing returns a *view* that shares memory with the original, so writing through it mutates the source; use `.copy()` for an independent array. C-order (row-major, the default) and Fortran-order (column-major) change which axis is contiguous and therefore which traversals are fastest.

```python
a = np.arange(12).reshape(3, 4)
print(a.strides)          # bytes to step along (rows, cols)
b = a[:, 1]               # a view, not a copy
b[0] = 999                # this also changes a[0, 1]
```
:::

:::{admonition} Going deeper: vectorisation vs loops
:class: seealso dropdown
Vectorised array operations run in compiled code and are typically one to two orders of magnitude faster than an equivalent python loop. You can measure it in a notebook:

```python
big = np.random.default_rng(0).random(1_000_000)
%timeit big + 1.0                       # vectorised
%timeit [x + 1.0 for x in big]          # python loop, much slower
```

The gap widens with array size. Reach for the array expression first; drop to a loop only when no vectorised form exists.
:::

:::{admonition} Going deeper: basic linear algebra
:class: seealso dropdown
numpy covers the everyday linear algebra a model needs.

```python
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
print(A @ b)                 # matrix-vector product (also np.matmul)
print(np.linalg.solve(A, b)) # solve A x = b without inverting A
```

Prefer `np.linalg.solve` over forming `np.linalg.inv(A) @ b`: it is more accurate and faster.
:::

:::{admonition} Going deeper: bigger-than-memory arrays with dask
:class: seealso dropdown
When a field is too large for memory, `dask.array` exposes the same numpy interface over *chunks*, building a task graph that runs only when you call `.compute()`.

```python
import dask.array as da
x = da.from_array(np.arange(1_000_000), chunks=100_000)
result = (x + 1).mean()      # lazy: nothing computed yet
print(result.compute())      # runs the graph, chunk by chunk
```

This previews the lazy, chunked model that xarray uses for climate-scale datasets in later subchapters.
:::

:::{admonition} Takeaways
:class: danger
- An array carries a `shape` and an `ndim`; create it from data or with constructors like `zeros`, `ones`, `arange`, and `linspace`.
- Index and slice with `[row, col]`; select with boolean masks; views share memory, so copy when you need independence.
- Vectorise: write array expressions instead of element loops, and use broadcasting to combine compatible shapes.
- Reduce along an explicit `axis` (`mean`, `sum`, `min`, `max`, `argmin`, `argmax`); reshape and stack to reorganise data.
- Use `np.where` to choose element-wise, NaN-aware ops for missing data, and `np.interp` to fill 1D gaps.
- Watch dtype on preallocation: `np.zeros_like(int_array)` truncates float results silently — let numpy promote to float.
- *(going deeper)* Every array has a dtype (`float64` default, `float32` half the size and coarser); many operations exist as both a method and an `np.` function.
:::

## Resources

- [Python Data Science Handbook — Introduction to NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html) — free online; thorough coverage of arrays, broadcasting, masking, and ufuncs.
- [Scientific Python Lectures — NumPy](https://lectures.scientific-python.org/intro/numpy/index.html) — a concise, research-oriented tour of array creation, operations, and reductions.